# 部位ごとに before / after 候補を選ぶ

`video_before_after_pair.ipynb` で抽出済みのフレームを再利用し、**目の下の質感・唇・頬を別々に**候補選定します。

- 目の下の質感: 顔サイズ比を 1.03 以下に制限
- 唇・頬: 顔サイズ比を 1.10 以下に制限
- 各部位: 検出手の重なりを 1% 以下に制限

候補がなくても閾値は自動で緩めません。手検出の見逃し、道具、影、髪は残り得るため、`report.html` を必ず目視します。


In [6]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
marker = Path('analysis/match_video_regions.py')
if (cwd / marker).is_file():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('repo root:', Path.cwd())


repo root: C:\Users\mail\work\ikiikimake


## 設定


In [7]:
VIDEO = Path('makeup0923.mp4')
OUTPUT = Path('outputs') / VIDEO.stem
SCAN_MANIFEST = OUTPUT / 'scan_manifest.json'
MODEL_DIR = REPO_ROOT / 'models'
TOP = 10
DIVERSITY_SECONDS = 15.0

# report.html を目視してから必要な部位だけ rank を入れます。
APPROVED_REGION_RANKS = {
    'eye_texture': 1,
    'lips': None,
    'cheeks': None,
}
if not VIDEO.is_file():
    raise FileNotFoundError(VIDEO)
if not SCAN_MANIFEST.is_file():
    raise FileNotFoundError(f'先に video_before_after_pair.ipynb を実行してください: {SCAN_MANIFEST}')
if not (MODEL_DIR / 'hand_landmarker.task').is_file():
    raise FileNotFoundError(MODEL_DIR / 'hand_landmarker.task')


## 部位別に手の重なりを再計測して候補を順位付け

既存のROI抽出は頬・額について手の重なりを持っていますが、唇・目の下については持っていません。ここでは保存済み元フレームに対して手検出を再実行し、各部位との重なりを測ります。


In [8]:
import hashlib
import json
from dataclasses import asdict
from importlib.metadata import version

from analysis.extract_face_rois import RoiConfig
from analysis.match_video_regions import (
    REGION_RULES,
    rank_region_pairs,
    scan_region_occlusion,
    sha256_file,
    write_region_report,
)

manifest = json.loads(SCAN_MANIFEST.read_text(encoding='utf-8'))
video_info = manifest.get('video')
if not isinstance(video_info, dict):
    raise ValueError('scan_manifest.json に video がありません。')
if Path(video_info.get('path', '')).resolve() != VIDEO.resolve():
    raise RuntimeError('scan_manifest.json の動画と VIDEO が一致しません。')
if sha256_file(VIDEO) != video_info.get('sha256'):
    raise RuntimeError('VIDEO の SHA-256 が scan_manifest.json と一致しません。')
records = manifest.get('records')
if not isinstance(records, list) or not records:
    raise ValueError('scan_manifest.json に records がありません。')
config = manifest.get('config', {})
split_seconds = float(config['split_seconds'])
min_gap_seconds = float(config['min_gap_seconds'])

# 手検出そのものは再実行ごとに微小差が出る可能性があるため、
# 既存runの再利用判定には検出結果ではなく「入力・実装・モデル・設定」だけを使う。
roi_config = RoiConfig()
hand_model_path = MODEL_DIR / 'hand_landmarker.task'
hand_model_spec = {
    'path': str(hand_model_path.resolve()),
    'sha256': sha256_file(hand_model_path),
    'mediapipe_version': version('mediapipe'),
    'settings': {
        'min_hand_detection_confidence': roi_config.min_hand_detection_confidence,
        'min_hand_presence_confidence': roi_config.min_hand_presence_confidence,
        'hand_margin_face_fraction': roi_config.hand_margin_face_fraction,
    },
}
rules_spec = json.loads(json.dumps(
    {name: asdict(rule) for name, rule in REGION_RULES.items()},
    ensure_ascii=False,
    sort_keys=True,
))
fingerprint_spec = {
    'scan_manifest_sha256': sha256_file(SCAN_MANIFEST),
    'implementation_sha256': sha256_file(REPO_ROOT / 'analysis' / 'match_video_regions.py'),
    'hand_model': hand_model_spec,
    'rules': rules_spec,
    'top': TOP,
    'diversity_seconds': DIVERSITY_SECONDS,
}
# JSONへ保存したとき tuple は list になるので、比較前にJSON表現へ正規化する。
# これをしないと同じ設定でも Python上の tuple/list 差だけで不一致になる。
fingerprint_spec = json.loads(json.dumps(
    fingerprint_spec, ensure_ascii=False, sort_keys=True
))
fingerprint = hashlib.sha256(
    json.dumps(fingerprint_spec, sort_keys=True, ensure_ascii=False).encode('utf-8')
).hexdigest()
RUN_DIR = OUTPUT / 'region_pair_candidates' / fingerprint[:16]

if RUN_DIR.exists():
    saved_occ = RUN_DIR / 'region_occlusion.json'
    saved_match = RUN_DIR / 'region_matching.json'
    saved_report = RUN_DIR / 'report.html'
    saved_fingerprint = RUN_DIR / 'run_fingerprint.json'
    required = (saved_occ, saved_match, saved_report, saved_fingerprint)
    if not all(path.is_file() for path in required):
        missing = [str(path) for path in required if not path.is_file()]
        raise FileExistsError(f'不完全な既存出力があります: {RUN_DIR}; missing={missing}')

    run_meta = json.loads(saved_fingerprint.read_text(encoding='utf-8'))
    if run_meta.get('fingerprint') != fingerprint or run_meta.get('specification') != fingerprint_spec:
        raise RuntimeError('既存runの入力・実装・モデル・設定fingerprintが現在と一致しません。')

    # 同じ入力条件なら、保存済みの手検出結果をそのまま再利用する。
    # 再検出して浮動小数点レベルの差を比較しない。
    occlusion = json.loads(saved_occ.read_text(encoding='utf-8'))
    matching = json.loads(saved_match.read_text(encoding='utf-8'))
    if occlusion.get('hand_model') != hand_model_spec:
        raise RuntimeError('既存 region_occlusion.json の手モデル情報がfingerprintと一致しません。')
    saved_rules = json.loads(json.dumps(
        occlusion.get('rules'), ensure_ascii=False, sort_keys=True
    ))
    if saved_rules != rules_spec:
        raise RuntimeError(
            '既存 region_occlusion.json の部位ルールがfingerprintと一致しません。 '
            f'saved={saved_rules} current={rules_spec}'
        )
    expected_match_settings = {
        'split_seconds': split_seconds,
        'min_gap_seconds': min_gap_seconds,
        'color_used_for_ranking': False,
        'region_hand_overlap_used_as_gate': True,
        'region_scale_used_as_gate': True,
        'thresholds_validated': False,
    }
    if matching.get('settings') != expected_match_settings:
        raise RuntimeError('既存 region_matching.json の選定設定が現在のscan設定と一致しません。')
    if matching.get('top_k') != TOP or matching.get('diversity_seconds') != DIVERSITY_SECONDS:
        raise RuntimeError('既存 region_matching.json のTOP/diversity設定が現在と一致しません。')
    print('同じ部位別候補を再利用します:', RUN_DIR)
else:
    occlusion = scan_region_occlusion(records, MODEL_DIR)
    if occlusion.get('hand_model') != hand_model_spec:
        raise RuntimeError('再計測した手モデル情報が事前fingerprintと一致しません。')
    measured_rules = json.loads(json.dumps(
        occlusion.get('rules'), ensure_ascii=False, sort_keys=True
    ))
    if measured_rules != rules_spec:
        raise RuntimeError(
            '再計測した部位ルールが事前fingerprintと一致しません。 '
            f'measured={measured_rules} current={rules_spec}'
        )
    matching = rank_region_pairs(
        records,
        split_seconds,
        min_gap_seconds,
        occlusion,
        top_k=TOP,
        diversity_seconds=DIVERSITY_SECONDS,
    )
    report_path = write_region_report(RUN_DIR, records, matching, occlusion)
    (RUN_DIR / 'region_occlusion.json').write_text(
        json.dumps(occlusion, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    (RUN_DIR / 'region_matching.json').write_text(
        json.dumps(matching, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    (RUN_DIR / 'run_fingerprint.json').write_text(
        json.dumps({'fingerprint': fingerprint, 'specification': fingerprint_spec}, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8')
    print('部位別候補を作成しました:', RUN_DIR)

print('report:', RUN_DIR / 'report.html')


同じ部位別候補を再利用します: outputs\makeup0923\region_pair_candidates\7fd28a2c9b73b23c
report: outputs\makeup0923\region_pair_candidates\7fd28a2c9b73b23c\report.html


## 候補の要約


In [9]:
from IPython.display import HTML, display

rows = []
for region, data in matching['regions'].items():
    if data['ranked_pairs']:
        p = data['ranked_pairs'][0]
        gate = p['region_gate']
        rows.append(
            f'<tr><td>{data["label"]}</td><td>{p["before_time"]:.2f}</td><td>{p["after_time"]:.2f}</td>'
            f'<td>{p["score"]:.4f}</td><td>{p["terms"]["face_scale_ratio"]:.4f}</td>'
            f'<td>{gate["before_hand_overlap_ratio"]:.2%}</td><td>{gate["after_hand_overlap_ratio"]:.2%}</td></tr>'
        )
    else:
        rows.append(f'<tr><td>{data["label"]}</td><td colspan="6">候補なし（閾値は自動緩和していません）</td></tr>')

display(HTML(
    '<table><thead><tr><th>部位</th><th>before秒</th><th>after秒</th><th>幾何差</th>'
    '<th>顔サイズ比</th><th>before手重なり</th><th>after手重なり</th></tr></thead><tbody>'
    + ''.join(rows) + '</tbody></table>'
))
print('必ず開いて目視:', RUN_DIR / 'report.html')


部位,before秒,after秒,幾何差,顔サイズ比,before手重なり,after手重なり
目の下の質感,240.99,1403.49,0.9303,1.0031,0.00%,0.00%
唇,222.51,1410.99,1.1004,1.0136,0.00%,0.00%
頬,240.99,1403.49,0.9303,1.0031,0.00%,0.00%


必ず開いて目視: outputs\makeup0923\region_pair_candidates\7fd28a2c9b73b23c\report.html


## 目視後に部位ごとの候補を固定

`APPROVED_REGION_RANKS` に rank を入れてこのセルを再実行します。既に別内容の `selected_region_pairs.json` がある場合は勝手に上書きせず停止します。


In [10]:
def selected_frame(record: dict) -> dict:
    image_path = Path(record['image_path'])
    roi_dir = Path(record['roi_dir'])
    required = {
        'roi_masks_sha256': roi_dir / 'roi_masks.npz',
        'roi_points_sha256': roi_dir / 'roi_points.json',
        'roi_overlay_sha256': roi_dir / 'roi_overlay.png',
    }
    for path in (image_path, *required.values()):
        if not path.is_file():
            raise FileNotFoundError(path)
    return {
        'frame_id': record['frame_id'],
        'timestamp_seconds': record['timestamp_seconds'],
        'image_path': str(image_path),
        'image_sha256': sha256_file(image_path),
        'roi_dir': str(roi_dir),
        **{name: sha256_file(path) for name, path in required.items()},
    }

by_id = {record['frame_id']: record for record in records}
if len(by_id) != len(records):
    raise ValueError('scan_manifest.json に重複 frame_id があります。')

selections = {}
for region, rank in APPROVED_REGION_RANKS.items():
    if rank is None:
        continue
    if isinstance(rank, bool) or not isinstance(rank, int):
        raise TypeError(f'{region}: rank は整数または None にしてください。')
    pairs = matching['regions'][region]['ranked_pairs']
    if not 1 <= rank <= len(pairs):
        raise ValueError(f'{region}: rank は 1..{len(pairs)} の範囲です。')
    p = pairs[rank - 1]
    selections[region] = {
        'rank': rank,
        'score': p['score'],
        'region_gate': p['region_gate'],
        'before': selected_frame(by_id[p['before_id']]),
        'after': selected_frame(by_id[p['after_id']]),
    }

if not selections:
    print('まだ固定していません。report.html を目視して APPROVED_REGION_RANKS を設定してください。')
else:
    selected = {
        'schema_version': 1,
        'video_path': str(VIDEO.resolve()),
        'video_sha256': video_info['sha256'],
        'region_candidate_run': str(RUN_DIR.resolve()),
        'selections': selections,
    }
    selected_path = OUTPUT / 'selected_region_pairs.json'
    if selected_path.exists():
        existing = json.loads(selected_path.read_text(encoding='utf-8'))
        if existing != selected:
            raise FileExistsError(f'別内容の selected_region_pairs.json が既にあります: {selected_path}')
        print('同じ部位別ペアが固定済みです:', selected_path)
    else:
        selected_path.write_text(json.dumps(selected, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        print('固定しました:', selected_path)


同じ部位別ペアが固定済みです: outputs\makeup0923\selected_region_pairs.json
